# DALIL : évaluation, fusion et GGUF d'un adaptateur LoRA déjà entraîné (variante D)

Reprend l'adaptateur produit par le carnet 03 (jeu de données Kaggle `dalil-lora-adapter`, envoyé par `python ml/kaggle/kaggle_run.py adapter`) sans refaire les 3 h d'entraînement. GPU T4, Internet activé.

In [ ]:
import glob, json, os, random, shutil, time
from collections import Counter
# Le jeu de données « dalil-pipeline » est monté sous /kaggle/input (chemin variable selon Kaggle) :
# on cherche le dossier qui contient backend/ et data/.
INPUT = next(root for root, dirs, _ in os.walk("/kaggle/input") if "backend" in dirs and "data" in dirs)
REPO = "/kaggle/working/dalil"
if not os.path.exists(REPO):
    shutil.copytree(INPUT, REPO)
os.chdir(REPO)
print(sorted(os.listdir(REPO)))

In [ ]:
!pip install -q -U "transformers>=4.51" peft trl bitsandbytes accelerate datasets
!pip uninstall -y -q torchao

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
NAME = "dalil-" + MODEL_ID.split("/")[-1].lower()
OUT = f"/kaggle/working/{NAME}"
N_EVAL, MAX_EVAL_MIN = 100, 40
RUN_EVAL = False   # déjà mesuré le 15/11/2026 : JSON valide 79/79, panne correcte en tête 50/79, candidats respectés 79/79
ADAPTER = next(root for root, _, files in os.walk("/kaggle/input") if "adapter_config.json" in files)
print("adaptateur :", ADAPTER)

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

data = load_dataset("json", data_files={"val": "data/processed/llm/val.jsonl"})
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
if RUN_EVAL:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, dtype=torch.float16)
    model = PeftModel.from_pretrained(model, ADAPTER).eval()

In [ ]:
if RUN_EVAL:
    # Évaluation sur la validation : JSON valide, panne correcte en tête, aucune panne hors des candidats
    import re
    def generate(messages, max_new_tokens=512):
        prompt = tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

    ok_json = ok_top1 = ok_cands = 0
    samples = []
    t_eval = time.time()
    val = data["val"].select(range(min(N_EVAL, len(data["val"]))))
    for i, ex in enumerate(val):
        gold = json.loads(ex["messages"][2]["content"])["ranking"][0]["fault"]
        cands = {c["slug"] for c in json.loads(ex["messages"][1]["content"])["candidats"]}
        text = generate(ex["messages"])
        if i < 3:
            print(f"--- réponse {i} (attendu : {gold}) ---\n{text[:800]}", flush=True)
        m = re.search(r"\{.*\}", text, re.S)
        try:
            ranking = json.loads(m.group(0))["ranking"]
            ok_json += 1
            ok_top1 += ranking[0]["fault"] == gold
            ok_cands += all(r["fault"] in cands for r in ranking)
        except Exception:
            pass
        samples.append({"gold": gold, "output": text})
        if time.time() - t_eval > MAX_EVAL_MIN * 60:
            print(f"Budget d'évaluation atteint après {i + 1} exemples", flush=True)
            val = val.select(range(i + 1))
            break
        if i % 10 == 0:
            print(f"évaluation {i}/{len(val)} ({time.time() - t_eval:.0f} s)", flush=True)
    n = len(val)
    result = {"n": n, "json_valide": ok_json, "top1": ok_top1, "candidats_respectes": ok_cands}
    print(f"JSON valide {ok_json}/{n}, panne correcte en tête {ok_top1}/{n}, candidats respectés {ok_cands}/{n}")
    with open(f"/kaggle/working/{NAME}-evaluation.json", "w", encoding="utf-8") as fh:
        json.dump({"resultat": result, "reponses": samples}, fh, ensure_ascii=False, indent=1)
    del model; torch.cuda.empty_cache()

In [ ]:
# Fusion de l'adaptateur dans le modèle de base (float16, sur processeur)
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained(f"/tmp/merged", safe_serialization=True)
del base, merged
# Fichiers du tokenizer d'origine : ceux réécrits par transformers 5 font échouer convert_hf_to_gguf
# (« 'list' object has no attribute 'keys' »).
from huggingface_hub import snapshot_download
src = snapshot_download(MODEL_ID, allow_patterns=["tokenizer*", "vocab.json", "merges.txt", "special_tokens_map.json",
                                                  "generation_config.json"])
for f in os.listdir(src):
    if os.path.isfile(os.path.join(src, f)):
        shutil.copy(os.path.join(src, f), f"/tmp/merged/{f}")
print(sorted(os.listdir(f"/tmp/merged")))

In [ ]:
# Conversion en GGUF pour llama.cpp (processeur du portable)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!pip uninstall -y -q torchao
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/{NAME}-f16.gguf --outtype f16
!rm -rf /tmp/merged && df -h /tmp /kaggle/working
# llama-quantize refuse de recompresser un fichier déjà en q8_0 : on part du f16. Modèle fusionné et f16 (8 Go
# chacun) dans /tmp : /kaggle/working est limité à ~20 Go (« Not enough free space »).
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
!/kaggle/working/llama.cpp/build/bin/llama-quantize /tmp/{NAME}-f16.gguf /kaggle/working/{NAME}-q4_k_m.gguf Q4_K_M
!/kaggle/working/llama.cpp/build/bin/llama-quantize /tmp/{NAME}-f16.gguf /kaggle/working/{NAME}-q8_0.gguf Q8_0
!rm -f /tmp/{NAME}-f16.gguf
!rm -rf /kaggle/working/llama.cpp && ls -lh /kaggle/working/*.gguf